# Step 1 · Python for ML: NumPy, broadcasting and PyTorch basics

This notebook goes with the **Lesson** step of [Setup · Step 1](https://learn.devmtnaing.com/ai-engineer/setup/step-1). Each section has:

- a short example to run and look at,
- an **exercise**: replace the `...` with your code,
- a **check** cell with `assert`s. If it prints ✅, you got it.

Stuck? Each exercise has a hidden **hint and solution** below it. Try first, then peek.

**Run cells in order** (Shift+Enter). In Colab, *Runtime → Change runtime type → GPU* gives you a free NVIDIA GPU for the last section.

In [ ]:
import time
import numpy as np
import torch

print("numpy", np.__version__, "| torch", torch.__version__)

## 1. NumPy arrays and shapes

Math on a whole array at once, no loops. Every array has a `shape`: (rows, columns) for 2D.

In [ ]:
prices = np.array([100, 200, 300])
print(prices * 2)              # [200 400 600]

x = np.arange(6).reshape(2, 3)
print(x)
print("shape:", x.shape)       # (2, 3)

print(x[0])        # first row       [0 1 2]
print(x[1, 2])     # row 1, col 2    5
print(x[:, 0])     # every row, col 0  [0 3]
print(x[0, 1:3])   # row 0, cols 1-2   [1 2]

### Exercise 1 · indexing

Using `x` from above, get:
- `col1`: the whole second column (`[1, 4]`)
- `last_row`: the last row (`[3, 4, 5]`)
- `corner`: the bottom-right number (`5`)

In [ ]:
col1 = ...
last_row = ...
corner = ...

<details><summary>💡 Hint</summary>

`:` means every row. Negative indexes count from the end: `x[-1]` is the last row.

</details>

<details><summary>✅ Solution</summary>

```python
col1 = x[:, 1]
last_row = x[1]       # or x[-1]
corner = x[1, 2]      # or x[-1, -1]
```

</details>

In [ ]:
assert np.array_equal(col1, [1, 4]), col1
assert np.array_equal(last_row, [3, 4, 5]), last_row
assert corner == 5, corner
print("✅ indexing")

## 2. `sum` / `mean` along an axis

`axis=0` collapses the **rows** (one value per column). `axis=1` collapses the **columns** (one value per row). `keepdims=True` keeps the collapsed axis as size 1.

In [ ]:
data = np.array([[1., 2., 3.],
                 [4., 5., 6.],
                 [7., 8., 9.],
                 [10., 11., 12.]])           # 4 images × 3 colour channels
print(data.mean(axis=0), data.mean(axis=0).shape)              # per channel → (3,)
print(data.mean(axis=1), data.mean(axis=1).shape)              # per image   → (4,)
print(data.mean(axis=1, keepdims=True).shape)                  # (4, 1)

### Exercise 2 · axis

- `row_sums`: the sum of each row of `data`, shape `(4,)`
- `col_max`: the largest value in each column, shape `(3,)`

In [ ]:
row_sums = ...
col_max = ...

<details><summary>💡 Hint</summary>

One value per row means collapsing the columns: `axis=1`. One value per column means `axis=0`.

</details>

<details><summary>✅ Solution</summary>

```python
row_sums = data.sum(axis=1)
col_max = data.max(axis=0)
```

</details>

In [ ]:
assert row_sums.shape == (4,) and np.allclose(row_sums, [6, 15, 24, 33]), row_sums
assert col_max.shape == (3,) and np.allclose(col_max, [10, 11, 12]), col_max
print("✅ axis")

## 3. Broadcasting

Line shapes up **from the right**: each pair must be equal, or one of them 1 (or missing).

```
(4, 3) - (3,)    → 3 vs 3 ✓            one value per COLUMN, stretched down the rows
(4, 3) - (4,)    → 3 vs 4 ✗  ERROR
(4, 3) - (4, 1)  → 3 vs 1 ✓, 4 vs 4 ✓   one value per ROW, stretched across
```

In [ ]:
channel_mean = data.mean(axis=0)                 # (3,)
print((data - channel_mean).shape)               # (4, 3)

try:
    data - data.mean(axis=1)                     # (4, 3) - (4,)
except ValueError as e:
    print("ERROR:", e)

image_mean = data.mean(axis=1, keepdims=True)    # (4, 1)
print((data - image_mean).shape)                 # (4, 3)

### Exercise 3 · centre each image

Subtract **each image's own mean** (each row's mean) from `data`, so every row of `centered` averages to 0. Use broadcasting, no loops.

In [ ]:
centered = ...

<details><summary>💡 Hint</summary>

You need one value per row, shape `(4, 1)`. `keepdims=True` gives you exactly that.

</details>

<details><summary>✅ Solution</summary>

```python
centered = data - data.mean(axis=1, keepdims=True)
```

</details>

In [ ]:
assert centered.shape == (4, 3), centered.shape
assert np.allclose(centered.mean(axis=1), 0), centered.mean(axis=1)
assert np.allclose(centered[0], [-1, 0, 1]), centered[0]
print("✅ broadcasting")

**Quick check:** what shape is `np.ones((8, 1)) + np.ones((1, 5))`? Guess, then run it.

In [ ]:
print((np.ones((8, 1)) + np.ones((1, 5))).shape)

## 4. Build · matrix multiplication: loops vs `@`

Shape rule: `[m, k] @ [k, n] → [m, n]`. The inner numbers must match.

### Exercise 4 · write it with three loops

Fill in `matmul_loops(A, B)`: `C[i, j]` is the sum over `k` of `A[i, k] * B[k, j]`. Use plain Python loops: that's the point.

In [ ]:
def matmul_loops(A, B):
    m, k = A.shape
    k2, n = B.shape
    assert k == k2, "inner dimensions must match"
    C = np.zeros((m, n))
    ...
    return C

<details><summary>💡 Hint</summary>

Loop `i` over rows of A, `j` over columns of B, and an inner loop that adds up `A[i, p] * B[p, j]`.

</details>

<details><summary>✅ Solution</summary>

```python
def matmul_loops(A, B):
    m, k = A.shape
    k2, n = B.shape
    assert k == k2, "inner dimensions must match"
    C = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            total = 0.0
            for p in range(k):
                total += A[i, p] * B[p, j]
            C[i, j] = total
    return C
```

</details>

In [ ]:
A = np.random.rand(3, 4)
B = np.random.rand(4, 2)
assert matmul_loops(A, B).shape == (3, 2)
assert np.allclose(matmul_loops(A, B), A @ B)
print("✅ matmul_loops matches A @ B")

### Time it

Start at 128 × 128. The course asks for **512 × 512**: set `n = 512` once it works. In plain Python that takes roughly 15 seconds on an Apple Silicon Mac (longer on Colab's CPU), so be patient. That *is* the lesson.

In [ ]:
n = 128          # then try 512
A = np.random.rand(n, n)
B = np.random.rand(n, n)

t0 = time.perf_counter(); C1 = matmul_loops(A, B); t_loops = time.perf_counter() - t0
t0 = time.perf_counter(); C2 = A @ B;              t_numpy = time.perf_counter() - t0

assert np.allclose(C1, C2)
print(f"{n}×{n}  loops: {t_loops:.3f} s   numpy: {t_numpy * 1000:.3f} ms   → {t_loops / t_numpy:,.0f}× faster")

## 5. Build · numerically stable softmax

`softmax(x) = exp(x) / sum(exp(x))`, applied to **each row** of a `[batch, classes]` array. Each row becomes probabilities that add up to 1.

The naive version breaks on big numbers:

In [ ]:
def softmax_naive(x):
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)

print(softmax_naive(np.array([[2.0, 1.0, 0.1]])))      # fine
print(softmax_naive(np.array([[1000.0, 999.0]])))      # nan: exp(1000) overflows

### Exercise 5 · make it stable

Subtract each row's **max** before `exp`. The answer doesn't change (the shift cancels in the division), but the biggest number becomes `exp(0) = 1`, so nothing overflows.

In [ ]:
def softmax(x):
    # x: shape (batch, classes)
    ...

<details><summary>💡 Hint</summary>

`x.max(axis=1, keepdims=True)` has shape `(batch, 1)`, so `x - that` broadcasts across each row. Then do the naive version.

</details>

<details><summary>✅ Solution</summary>

```python
def softmax(x):
    # x: shape (batch, classes)
    x = x - x.max(axis=1, keepdims=True)    # (batch, 1) broadcasts across the row
    e = np.exp(x)
    return e / e.sum(axis=1, keepdims=True)
```

</details>

In [ ]:
logits = np.random.randn(4, 10) * 5
p = softmax(logits)
assert p.shape == (4, 10)
assert np.allclose(p.sum(axis=1), 1), "each row must add up to 1"

big = softmax(np.array([[1000.0, 999.0], [-1000.0, -1000.0]]))
assert np.all(np.isfinite(big)), big
assert np.allclose(big[1], [0.5, 0.5])

# matches PyTorch's own softmax
assert np.allclose(p, torch.softmax(torch.from_numpy(logits), dim=1).numpy())
print("✅ stable softmax matches torch.softmax")

## 6. PyTorch tensors

Same ideas as NumPy, plus GPUs and autograd. `reshape` keeps the same numbers (rows × cols must equal the count), and `-1` means "work it out".

In [ ]:
t = torch.arange(6)
print(t, t.shape)
print(t.reshape(2, 3))
print(t.reshape(-1, 1).shape)    # (6, 1)

try:
    t.reshape(4, 2)
except RuntimeError as e:
    print("ERROR:", e)

### Exercise 6 · reshape

From `torch.arange(12)`, make:
- `grid`: shape `(3, 4)`
- `col`: a single column, shape `(12, 1)`, using `-1` for the 12

In [ ]:
grid = ...
col = ...

<details><summary>💡 Hint</summary>

`.reshape(rows, cols)`. For the column, one column means the second number is 1.

</details>

<details><summary>✅ Solution</summary>

```python
grid = torch.arange(12).reshape(3, 4)
col = torch.arange(12).reshape(-1, 1)
```

</details>

In [ ]:
assert grid.shape == (3, 4) and grid[1, 0] == 4, grid
assert col.shape == (12, 1), col.shape
print("✅ reshape")

## 7. NumPy ↔ PyTorch: shared memory

`torch.from_numpy(a)` **shares** memory with `a`; `torch.tensor(a)` makes a **copy**. Predict both prints before you run it.

In [ ]:
a = np.array([1, 2, 3])
shared = torch.from_numpy(a)
copied = torch.tensor(a)
a[0] = 100
print("shared:", shared)    # ?
print("copied:", copied)    # ?

## 8. Devices

Tensors live on `cpu`, `cuda` (NVIDIA, e.g. Colab GPU) or `mps` (Apple Silicon). Maths only works between tensors on the **same** device. Pick it once at the top.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("using", device)

x = torch.tensor([1.0, 2.0], device=device)
y = torch.tensor([3.0, 4.0], device=device)
print(x + y)

## 9. Training: gradient descent with autograd

The target is 5; `w` starts at 3; `loss = (w - 5) ** 2`. `loss.backward()` fills in `w.grad`, we step opposite it, then **zero** the gradient (it accumulates otherwise).

In [ ]:
w = torch.tensor(3.0, requires_grad=True)
lr = 0.1
for step in range(3):
    loss = (w - 5) ** 2
    loss.backward()
    with torch.no_grad():
        w -= lr * w.grad
    w.grad.zero_()
    print(step + 1, round(w.item(), 4))    # 3.4, 3.72, 3.976

### Exercise 7 · train until it arrives

Write `train_until(lr)`: start `w` at 3.0 **on `device`** (pass `device=device` to `torch.tensor`; `.to(device)` afterwards makes a copy with no `.grad`), loop with `while` until `w` is within `0.001` of 5, and **return the number of steps**.

Then try different learning rates. What happens at `lr=1.0`? (Don't run that one without a step limit!)

In [ ]:
def train_until(lr, target=5.0, tol=0.001):
    w = ...
    steps = 0
    ...
    return steps

<details><summary>💡 Hint</summary>

Start from the loop above. Swap `for` for `while abs(w.item() - target) > tol:` and add `steps += 1` inside.

</details>

<details><summary>✅ Solution</summary>

```python
def train_until(lr, target=5.0, tol=0.001):
    w = torch.tensor(3.0, requires_grad=True, device=device)
    steps = 0
    while abs(w.item() - target) > tol:
        loss = (w - target) ** 2
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
        w.grad.zero_()
        steps += 1
    return steps
```

</details>

In [ ]:
steps = train_until(0.1)
print("lr=0.1 →", steps, "steps")
assert steps == 35, steps
assert train_until(0.3) < steps, "a bigger (but not too big) learning rate should need fewer steps"
print("✅ training loop")

## Done 🎉

You've run everything in Step 1. Before the checkpoint, make sure you can:

- predict the shape of a broadcast or a matmul **before** running it,
- write the stable softmax from memory,
- explain why `zero_()` matters and what a too-big learning rate does.

Then do the **Quiz** and **Checkpoint** steps on the site.